# Plot GeoCSV measurements

Read the included P0006 data and make two plots with pandas and Matplotlib.
These examples run offline and perform analysis after parsing. The
[rc2 subset](../data/fixtures/schema-1.0.0-rc2/README.md) contains the
first 300 rows of the producer export.

Companions: [Markdown](06_plotting.md) · [Notebook](06_plotting.ipynb).

## Setup

From the cloned repository, activate your environment and install the packages:

```bash
source .venv/bin/activate
python -m pip install . matplotlib
```

See [installation and notebook setup](../docs/installation.md) if needed. Run the
Python examples in order from the repository root, `docs/`, or `tutorials/`.

## 1. Read the data



In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from mermaid import geocsv

root = Path.cwd()
if root.name in {"docs", "tutorials"}:
    root = root.parent

records = geocsv.read(root / "data/fixtures/schema-1.0.0-rc2/P0006.geocsv")[0]



## 2. Plot pressure measurements

Select pressure measurements from the first two weeks. Plot the source values
in mbar, without converting them to depth:



In [ ]:
pressure = records.loc[
    records["type"].eq("pressure")
    & records["measured_time"].ge("2018-06-27")
    & records["measured_time"].lt("2018-07-11")
].dropna(subset=["measured_pressure"]).sort_values("measured_time")

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(pressure["measured_time"], pressure["measured_pressure"])
ax.set(xlabel="Time (UTC)", ylabel="Gauge pressure (mbar)",
       title="P0006 pressure measurements")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()



## 3. Plot GPS latitude over time

GPS records describe surface fixes. This plot selects the fixes in the included prefix:



In [ ]:
gps = records.loc[
    records["type"].eq("gps")
].dropna(subset=["measured_latitude"]).sort_values("measured_time")

fig, ax = plt.subplots(figsize=(9, 3))
ax.scatter(gps["measured_time"], gps["measured_latitude"], s=10)
ax.set(xlabel="Time (UTC)", ylabel="Latitude (degrees north)",
       title="P0006 GPS fixes")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()



Use `fig.savefig("p0006_gps.png", dpi=180)` to save the current figure.
Further analysis is downstream of parsing; choose coordinate systems and any
receiver (station) pressure-to-depth model explicitly for the scientific
question. The optional SAC projection uses the documented approximation
100 mbar per metre of MERMAID station depth in `stdp`.
